# 09 · Attention and Transformers

**Part IV — Sequences and language** · Estimated time: ~150 min

## Learning objectives

- Derive scaled dot-product attention from the idea of a soft dictionary lookup and explain, with a numerical experiment, why the scores are divided by $\sqrt{d_k}$.
- Implement padding and causal masks and multi-head attention from scratch, and verify the latter against `nn.MultiheadAttention`.
- Explain why attention is permutation-equivariant and how sinusoidal encodings, learned embeddings and rotary embeddings (RoPE) inject positional information.
- Assemble a pre-LN Transformer block and count its parameters and FLOPs.
- Build a GPT-style decoder-only language model with weight tying and train it from scratch with AdamW, warmup–cosine scheduling and gradient clipping.
- Generate text with greedy, temperature, top-$k$ and top-$p$ decoding, and implement a KV-cache that makes generation linear instead of quadratic.
- Reason about scaling laws, the cost of long contexts, and the encoder-only, encoder–decoder and decoder-only members of the Transformer family.

## Prerequisites

- Notebook 03 · Tensors and Autograd (einsum, broadcasting, reshapes)
- Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization (AdamW, learning-rate schedules, LayerNorm, residual connections)
- Notebook 08 · Sequence Models: RNNs, LSTMs and GRUs (language modelling, teacher forcing, the seq2seq bottleneck)

## Contents

1. Attention as a soft dictionary lookup
2. Masking
3. Multi-head attention
4. Positional information
5. The Transformer block
6. Build a GPT
7. Generation and the KV-cache
8. Scaling
9. The Transformer family

Exercises · Summary · Further reading

In [ ]:
import math, os, random, time
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

## 1. Attention as a soft dictionary lookup

### 1.1 From hard lookup to soft lookup

Notebook 08 · Sequence Models: RNNs, LSTMs and GRUs ended with a diagnosis: a recurrent encoder must squeeze a whole input sequence into one fixed-size vector, and a decoder must then recover, from that vector alone, whatever it needs at each step. The cure proposed by Bahdanau, Cho and Bengio (2015) was to keep *all* the encoder states and let the decoder look up the relevant ones at every step. The Transformer (Vaswani et al., 2017) took the idea to its conclusion: if lookup works so well, why keep the recurrence at all? Its only mechanism for moving information between positions is **attention**, applied repeatedly, and everything else is a position-wise MLP.

Start from a Python dictionary. It stores pairs `key -> value`; a lookup with a `query` finds the key that *equals* the query and returns its value. Attention is the differentiable version of this operation. Keys, values and queries are vectors; instead of testing equality, we compute a **similarity score** between the query and every key; instead of returning one value, we return a **weighted average of all values**, with weights obtained by passing the scores through a softmax. If one key matches the query much better than all others, the softmax puts nearly all the weight on it and we recover the hard lookup; if several keys match partially, we get a blend. Because every step is differentiable, the network can *learn* what to store as keys and values and what to ask for as queries.

Concretely, with a query $q \in \mathbb{R}^{d_k}$, keys $k_1, \dots, k_n \in \mathbb{R}^{d_k}$ and values $v_1, \dots, v_n \in \mathbb{R}^{d_v}$:

$$
\alpha_j = \frac{\exp(s_j)}{\sum_{l=1}^{n} \exp(s_l)}, \qquad s_j = \frac{q^{\top} k_j}{\sqrt{d_k}}, \qquad \text{output} = \sum_{j=1}^{n} \alpha_j v_j .
$$

The next cell performs exactly this computation on five keys with a query that is close to key 2 and somewhat similar to key 4, and then shows what happens as the query becomes an exact match.

In [ ]:
torch.manual_seed(SEED)
d_k = 4
keys = torch.randn(5, d_k)
values = torch.arange(5.0)[:, None] * 10          # value j is the vector [10*j]; easy to read off in a blend
query = 0.8 * keys[2] + 0.4 * keys[4]             # a query that resembles key 2 and, less so, key 4

scores = keys @ query / math.sqrt(d_k)            # one similarity score per key
weights = F.softmax(scores, dim=0)
output = weights @ values
print("scores :", scores.cpu().numpy().round(2))
print("weights:", weights.cpu().numpy().round(3), "  (sum = %.3f)" % weights.sum().item())
print("output :", output.cpu().numpy().round(2), "<- a blend of value 2 (=20) and value 4 (=40)")

sharp_query = 4.0 * keys[2]                       # scale the query up: the softmax approaches a hard argmax
w_sharp = F.softmax(keys @ sharp_query / math.sqrt(d_k), dim=0)
print("weights for a sharper query:", w_sharp.cpu().numpy().round(3), "-> output", (w_sharp @ values).cpu().numpy().round(2))

### 1.2 Queries, keys and values from the same sequence

In a Transformer the three roles are played by *linear projections of the token representations*. Given a sequence $X \in \mathbb{R}^{T \times d}$ (one row per position), we compute

$$
Q = X W_Q, \qquad K = X W_K, \qquad V = X W_V,
$$

with learned matrices $W_Q, W_K \in \mathbb{R}^{d \times d_k}$ and $W_V \in \mathbb{R}^{d \times d_v}$. Every position issues a query, offers a key, and offers a value. Stacking all queries and keys, the scores form a $T \times T$ matrix and the whole operation is two matrix products around a row-wise softmax,

$$
\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\!\left(\frac{Q K^{\top}}{\sqrt{d_k}}\right) V .
$$

Row $i$ of the softmax output holds the weights with which position $i$ reads from every position $j$. This is **self-attention**: the sequence attends to itself. When queries come from one sequence and keys and values from another (a decoder reading an encoder), it is called **cross-attention**; the formula is identical.

### 1.3 Why divide by $\sqrt{d_k}$: the variance argument

The scaling looks like a detail; it is essential. Suppose the entries of $q$ and $k$ are independent with mean zero and variance one (a reasonable picture after LayerNorm and a random projection). The dot product $q^{\top} k = \sum_{i=1}^{d_k} q_i k_i$ is a sum of $d_k$ independent terms, each with mean zero and variance $\mathbb{E}[q_i^2]\mathbb{E}[k_i^2] = 1$, so

$$
\operatorname{Var}\!\left(q^{\top} k\right) = d_k, \qquad \operatorname{Var}\!\left(\frac{q^{\top} k}{\sqrt{d_k}}\right) = 1 .
$$

Without the division, scores have a standard deviation of $\sqrt{d_k}$: for $d_k = 256$ a typical score is about $\pm 16$ units, two scores easily differ by that much, and $\exp(16) \approx 9 \times 10^6$, so the softmax becomes a hard argmax. A saturated softmax is bad for two reasons. The output stops being a blend (attention loses its ability to combine information), and the gradient of the softmax, $\partial \alpha_j / \partial s_l = \alpha_j(\delta_{jl} - \alpha_l)$, is essentially zero when one $\alpha$ is 1 and the rest are 0, so the projections $W_Q, W_K$ stop learning. The cell below measures the score variance and the average *maximum* attention weight (a saturation indicator) as $d_k$ grows, with and without the scaling, over 16 random keys.

In [ ]:
torch.manual_seed(SEED)
dims = [4, 16, 64, 256, 1024]
n_keys, n_trials = 16, 2000
rows = []
for d in dims:
    q = torch.randn(n_trials, d)
    k = torch.randn(n_trials, n_keys, d)
    raw = torch.einsum("nd,nkd->nk", q, k)                       # unscaled scores, (n_trials, n_keys)
    scaled = raw / math.sqrt(d)
    rows.append((d, raw.var().item(), scaled.var().item(),
                 F.softmax(raw, dim=-1).max(dim=-1).values.mean().item(),
                 F.softmax(scaled, dim=-1).max(dim=-1).values.mean().item()))
print(f"{'d_k':>5} | {'var(raw)':>9} {'var(scaled)':>12} | {'max weight raw':>15} {'max weight scaled':>18}")
for d, vr, vs, mr, ms in rows:
    print(f"{d:5d} | {vr:9.1f} {vs:12.2f} | {mr:15.3f} {ms:18.3f}")

fig, ax = plt.subplots()
ax.plot(dims, [r[3] for r in rows], marker="o", label="unscaled $q^\\top k$")
ax.plot(dims, [r[4] for r in rows], marker="s", label="scaled $q^\\top k / \\sqrt{d_k}$")
ax.axhline(1 / n_keys, color="gray", linestyle=":", label=f"uniform (1/{n_keys})")
ax.set_xscale("log"); ax.set_xlabel("$d_k$"); ax.set_ylabel("mean of the largest attention weight")
ax.set_title("Softmax saturation with and without scaling (16 random keys)")
ax.legend(); plt.tight_layout(); plt.show()

The variance of the unscaled scores grows exactly like $d_k$, and the largest attention weight climbs towards 1 as the dimension increases: at $d_k = 1024$ random queries and keys already produce nearly one-hot attention, before any learning has happened. With the scaling the variance stays at 1 and the largest weight stays modest, leaving the softmax in the regime where it blends and where its gradients are alive. The network can still *learn* to make attention sharp, by growing the norms of its projections, but it starts from a soft state and gets there when the data require it.

### 1.4 Implementation and a first look at attention weights

The from-scratch function below handles batches and extra leading dimensions (we will add a head dimension later) and accepts an optional boolean mask in which `True` means "may attend". We check it against PyTorch's fused `F.scaled_dot_product_attention`, then visualise the weights of a single-head self-attention over a toy sentence. The embeddings are random and the projections untrained, so the pattern is not meaningful; the point is to learn to *read* such a heatmap: rows are queries, columns are keys, each row sums to one, and identical tokens (the two occurrences of "the") produce identical rows because they issue identical queries.

In [ ]:
def attention(q, k, v, mask=None):
    """Scaled dot-product attention.

    q: (..., T_q, d_k), k: (..., T_k, d_k), v: (..., T_k, d_v); mask: bool, broadcastable to (..., T_q, T_k),
    True where the query may attend. Returns (output (..., T_q, d_v), weights (..., T_q, T_k))."""
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ v, weights


torch.manual_seed(SEED)
q, k, v = torch.randn(2, 5, 8), torch.randn(2, 7, 8), torch.randn(2, 7, 6)
out_ours, w_ours = attention(q, k, v)
out_ref = F.scaled_dot_product_attention(q, k, v)
print("matches F.scaled_dot_product_attention:", torch.allclose(out_ours, out_ref, atol=1e-6),
      "| rows sum to one:", torch.allclose(w_ours.sum(-1), torch.ones(2, 5)))

# A toy sentence with a shared embedding per word type and untrained projections.
sentence = "the cat sat on the mat".split()
vocab_toy = {w: i for i, w in enumerate(dict.fromkeys(sentence))}
d_model_toy = 8
emb_toy = torch.randn(len(vocab_toy), d_model_toy)
X = emb_toy[[vocab_toy[w] for w in sentence]]                       # (T, d)
W_Q, W_K, W_V = (torch.randn(d_model_toy, d_model_toy) / math.sqrt(d_model_toy) for _ in range(3))
_, A = attention(X @ W_Q, X @ W_K, X @ W_V)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(A.cpu().numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(sentence))); ax.set_xticklabels(sentence)
ax.set_yticks(range(len(sentence))); ax.set_yticklabels(sentence)
ax.set_xlabel("key position (read from)"); ax.set_ylabel("query position (reading)")
ax.set_title("Self-attention weights (untrained)"); ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.046); plt.tight_layout(); plt.show()
print("rows for the two occurrences of 'the' are identical:", torch.allclose(A[0], A[4]))

The identical rows for the two "the" tokens illustrate a limitation we will fix in Section 4: attention as defined so far has no notion of *where* a token is, only of *what* it is. First, though, we need masks.

> **Key idea.** Attention computes, for every query, a softmax over similarity scores with all keys and returns the corresponding weighted average of values. The $\sqrt{d_k}$ scaling keeps the softmax soft and its gradients alive at initialisation.

## 2. Masking

Attention over "all keys" is often too much. Two situations require excluding some keys, and both are handled the same way: set the corresponding score to $-\infty$ *before* the softmax, so that its weight becomes exactly $e^{-\infty} = 0$ and the remaining weights still sum to one.

**Padding masks.** Sequences in a batch are padded to a common length (Notebook 08 · Sequence Models: RNNs, LSTMs and GRUs). Pad positions carry no information and must not receive attention, so every query masks out the keys at padded positions. A padding mask is a boolean tensor of shape `(batch, T_k)`, broadcast to `(batch, 1, T_k)` or `(batch, 1, 1, T_k)` when heads are present.

**Causal masks.** A language model predicts token $t+1$ from tokens $1, \dots, t$. If, during training, position $t$ could attend to positions $t+1, \dots, T$, it would simply read the answer from the input and learn nothing. The causal (or "look-ahead") mask allows position $i$ to attend only to positions $j \le i$: a lower-triangular matrix of `True`. It is what makes a Transformer *decoder* autoregressive, and it is the only difference between a GPT-style decoder block and a BERT-style encoder block.

The two masks are combined with a logical AND. The cell builds both for a small batch, applies them through the `attention` function, and shows the resulting weight matrices for one sequence of length 4 padded to 6.

In [ ]:
def causal_mask(T, device=None):
    """(T, T) boolean mask: position i may attend to j <= i."""
    return torch.tril(torch.ones(T, T, dtype=torch.bool, device=device))


def padding_mask(lengths, T):
    """(batch, T) boolean mask: True at real tokens, False at padding."""
    return torch.arange(T, device=lengths.device)[None, :] < lengths[:, None]


torch.manual_seed(SEED)
B, T, d = 2, 6, 8
lengths = torch.tensor([4, 6])
x = torch.randn(B, T, d)
pad = padding_mask(lengths, T)                       # (B, T)   keys that are real
causal = causal_mask(T)                              # (T, T)
combined = causal[None, :, :] & pad[:, None, :]      # (B, T, T): AND of both

_, w_pad = attention(x, x, x, mask=pad[:, None, :])
_, w_causal = attention(x, x, x, mask=causal)
_, w_both = attention(x, x, x, mask=combined)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, w, title in zip(axes, [w_pad[0], w_causal[0], w_both[0]],
                        ["padding mask (length 4 of 6)", "causal mask", "padding AND causal"]):
    ax.imshow(w.cpu().numpy(), cmap="Blues", vmin=0, vmax=1)
    ax.set_title(title); ax.set_xlabel("key"); ax.set_ylabel("query"); ax.grid(False)
plt.tight_layout(); plt.show()
print("masked weights are exactly zero:", bool((w_both[0][~combined[0]] == 0).all()),
      "| every row still sums to one:", torch.allclose(w_both.sum(-1), torch.ones(B, T)))

In the first panel the last two columns are blank for every query: nobody reads from padding. In the second, the upper triangle is blank: nobody reads from the future. The third panel combines both. Note that the *queries* at padded positions (rows 4 and 5 of the first and third panels) still produce weights; their outputs are garbage that must be ignored downstream by the loss (`ignore_index`) or by pooling with a mask, exactly as in Notebook 08 · Sequence Models: RNNs, LSTMs and GRUs.

> **Common pitfall.** If a row is masked *entirely*, the softmax of a row of $-\infty$ is `nan`, and the `nan` propagates through the whole network. This happens with left-padding combined with a causal mask (the first query sees only a padded key), or with an all-pad sequence. Guard against it yourself, for example by right-padding or by letting a pad query attend to itself; do not rely on the library, because the treatment of fully masked rows differs between code paths and PyTorch versions (recent releases of `F.scaled_dot_product_attention` return a row of zeros for such a query, other paths return `nan`). Another classic mistake is the sign convention: our masks and `F.scaled_dot_product_attention` use `True` for "may attend", whereas `nn.MultiheadAttention` and `nn.Transformer` use `True` for "**masked out**". Always check the docstring of the API you are calling.

## 3. Multi-head attention

### 3.1 Motivation

A single attention head computes one weighted average per position. But a token may need to gather several *different* kinds of information at once: the previous word, the subject of the sentence, the matching opening bracket, the last occurrence of the same word. One softmax cannot point in several directions at once without blurring them all together. **Multi-head attention** runs $h$ attention operations in parallel, each with its own projections $W_Q^{(i)}, W_K^{(i)}, W_V^{(i)}$ of reduced size $d_h = d / h$, and concatenates their outputs:

$$
\text{head}_i = \operatorname{Attention}\!\left(X W_Q^{(i)}, X W_K^{(i)}, X W_V^{(i)}\right), \qquad
\operatorname{MultiHead}(X) = \left[\text{head}_1; \dots; \text{head}_h\right] W_O .
$$

Because each head works in $d/h$ dimensions, the total cost is the same as a single head of full width, and the output projection $W_O \in \mathbb{R}^{d \times d}$ mixes the heads. The whole thing is an *ensemble of lookups* whose results are combined linearly.

### 3.2 Implementation with reshapes

The implementation trick is to compute all heads with one matrix product and then *reshape* rather than loop. A projection produces `(batch, T, d)`; viewing it as `(batch, T, h, d_h)` and transposing to `(batch, h, T, d_h)` places the head dimension where our `attention` function treats it as an extra batch dimension. After attention, we transpose back, merge `(h, d_h)` into `d`, and apply $W_O$. The class accepts separate query and key/value inputs so that it also serves cross-attention.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head attention with the same parametrisation as nn.MultiheadAttention (batch_first)."""

    def __init__(self, d_model, n_heads, bias=True):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model, self.n_heads, self.d_head = d_model, n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=bias)
        self.W_k = nn.Linear(d_model, d_model, bias=bias)
        self.W_v = nn.Linear(d_model, d_model, bias=bias)
        self.W_o = nn.Linear(d_model, d_model, bias=bias)

    def forward(self, x_q, x_kv, mask=None):
        B, T_q, _ = x_q.shape
        T_k = x_kv.shape[1]
        q = self.W_q(x_q).view(B, T_q, self.n_heads, self.d_head).transpose(1, 2)    # (B, h, T_q, d_h)
        k = self.W_k(x_kv).view(B, T_k, self.n_heads, self.d_head).transpose(1, 2)   # (B, h, T_k, d_h)
        v = self.W_v(x_kv).view(B, T_k, self.n_heads, self.d_head).transpose(1, 2)
        out, weights = attention(q, k, v, mask)                                       # heads act as a batch dim
        out = out.transpose(1, 2).reshape(B, T_q, self.d_model)                       # concatenate the heads
        return self.W_o(out), weights                                                 # weights: (B, h, T_q, T_k)


torch.manual_seed(SEED)
mha = MultiHeadAttention(d_model=32, n_heads=4)
x = torch.randn(2, 7, 32)
y, w = mha(x, x, mask=causal_mask(7))
print("output:", tuple(y.shape), "| weights:", tuple(w.shape), "= (batch, heads, T_q, T_k)")
print("parameters:", sum(p.numel() for p in mha.parameters()), "= 4 * (d*d + d) =", 4 * (32 * 32 + 32))

### 3.3 Verification against `nn.MultiheadAttention`

PyTorch's module packs the three input projections into one `in_proj_weight` of shape `(3d, d)` (rows: query, key, value) with a matching `in_proj_bias`, and keeps `out_proj` as a separate `nn.Linear`. We copy those slices into our module, run both on the same input with the same causal mask (remembering the inverted mask convention), and compare outputs and head-averaged weights.

In [ ]:
torch.manual_seed(SEED)
d_model, n_heads, B, T = 32, 4, 2, 7
ref = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
ours = MultiHeadAttention(d_model, n_heads)
with torch.no_grad():
    Wq, Wk, Wv = ref.in_proj_weight.chunk(3, dim=0)
    bq, bk, bv = ref.in_proj_bias.chunk(3, dim=0)
    ours.W_q.weight.copy_(Wq); ours.W_q.bias.copy_(bq)
    ours.W_k.weight.copy_(Wk); ours.W_k.bias.copy_(bk)
    ours.W_v.weight.copy_(Wv); ours.W_v.bias.copy_(bv)
    ours.W_o.weight.copy_(ref.out_proj.weight); ours.W_o.bias.copy_(ref.out_proj.bias)

x = torch.randn(B, T, d_model)
mask = causal_mask(T)
y_ours, w_ours = ours(x, x, mask=mask)
y_ref, w_ref = ref(x, x, x, attn_mask=~mask, need_weights=True, average_attn_weights=True)   # True = masked OUT
print("max |output difference| :", (y_ours - y_ref).abs().max().item())
print("max |weights difference|:", (w_ours.mean(dim=1) - w_ref).abs().max().item(), "(head-averaged)")

The two agree to floating-point precision. From now on we know exactly what `nn.MultiheadAttention` does, and we will write our own attention modules inside the GPT because we need control over caching (Section 7).

### 3.4 `F.scaled_dot_product_attention` and FlashAttention

Our `attention` function materialises the $T \times T$ score matrix, applies the mask, takes the softmax, and multiplies by $V$. For $T = 4096$ and 32 heads that matrix alone occupies $4096^2 \times 32 \times 4$ bytes $\approx 2$ GB per sequence in float32, and reading and writing it dominates the run time, because attention is *memory-bound*: the arithmetic is cheap relative to the traffic between GPU memory and compute units. **FlashAttention** (Dao et al., 2022) computes exactly the same result without ever writing the full matrix: it processes keys in blocks, keeps a running softmax normaliser (the "online softmax" trick), and accumulates the output block by block in fast on-chip memory. The result is identical up to rounding, memory is linear in $T$, and wall-clock time drops by a factor of 2–4 on typical hardware.

`torch.nn.functional.scaled_dot_product_attention` is the single entry point to these kernels. It takes `(…, T, d)` tensors for `q, k, v`, an optional boolean or additive `attn_mask`, a `dropout_p`, and `is_causal=True` for the causal case (it then builds no mask at all), and it dispatches to FlashAttention, to a memory-efficient kernel, or to the plain math implementation depending on the device, dtype and mask. It never returns the attention weights, which is precisely what allows it to avoid materialising them. The cell times the two implementations on a moderately large problem; on a CPU the gap is modest, on a GPU with float16 it is large.

In [ ]:
def time_fn(fn, n_repeat=5):
    """Median wall-clock seconds of fn(), synchronising the GPU when needed."""
    times = []
    for _ in range(n_repeat):
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return float(np.median(times))


B, h, T, d_h = (4, 8, 512, 64) if FAST_RUN else (8, 8, 1024, 64)
q, k, v = (torch.randn(B, h, T, d_h, device=device) for _ in range(3))
mask = causal_mask(T, device=device)
with torch.no_grad():
    t_scratch = time_fn(lambda: attention(q, k, v, mask))
    t_sdpa = time_fn(lambda: F.scaled_dot_product_attention(q, k, v, is_causal=True))
    same = torch.allclose(attention(q, k, v, mask)[0], F.scaled_dot_product_attention(q, k, v, is_causal=True), atol=1e-4)
print(f"B={B}, heads={h}, T={T}, d_head={d_h} on {device.type}")
print(f"from-scratch attention: {t_scratch * 1e3:8.1f} ms | F.scaled_dot_product_attention: {t_sdpa * 1e3:8.1f} ms "
      f"| speed-up {t_scratch / t_sdpa:.1f}x | same result: {same}")
print(f"score matrix that the from-scratch version materialises: {B * h * T * T * 4 / 2**20:.0f} MiB (float32)")

Even on a CPU the fused kernel avoids the separate mask, softmax and matmul passes and is usually faster; on a GPU it is the difference between fitting a long context in memory or not. In the rest of the notebook the from-scratch `attention` is kept for anything that needs the weights (visualisation, verification), and `F.scaled_dot_product_attention` is used inside the model.

> **Going deeper.** With `attn_mask` given as a *float* tensor, the mask is *added* to the scores, which allows arbitrary biases: relative-position biases (Exercise 1), ALiBi slopes, or soft masks. A boolean mask is converted internally into a $0 / -\infty$ additive mask.

## 4. Positional information

### 4.1 Attention is permutation-equivariant

Look again at the attention formula. Each output row is a weighted average of value rows, with weights that depend only on dot products between the query and the keys. Nothing in it refers to *where* a token is: if we shuffle the rows of $X$ with a permutation matrix $P$, every query still meets the same set of keys and values, and the output rows are simply shuffled the same way,

$$
\operatorname{Attention}(PX) = P \operatorname{Attention}(X).
$$

This property is called **permutation equivariance**. The position-wise MLP and LayerNorm of a Transformer act on each row independently and are equivariant too, so a whole stack of Transformer blocks is a function of the *set* of tokens, not the sequence. "The dog bit the man" and "the man bit the dog" would receive the same representations up to reordering. The cell verifies this on our multi-head module.

In [ ]:
torch.manual_seed(SEED)
mha = MultiHeadAttention(d_model=16, n_heads=4)
x = torch.randn(1, 6, 16)
perm = torch.randperm(6)
y, _ = mha(x, x)
y_perm, _ = mha(x[:, perm], x[:, perm])
print("permutation:", perm.tolist())
print("Attention(P x) == P Attention(x):", torch.allclose(y[:, perm], y_perm, atol=1e-6))

The fix is to make the input rows position-dependent by adding (or otherwise injecting) a **positional encoding** $p_t$ to the token embedding $e_{x_t}$, so that the representation at position $t$ is $e_{x_t} + p_t$. Three designs dominate.

### 4.2 Sinusoidal encodings and the relative-shift property

The original Transformer used fixed sinusoids. For position $t$ and encoding dimension $d$, with pairs of dimensions $(2i, 2i+1)$ sharing a frequency $\omega_i = 10000^{-2i/d}$,

$$
p_{t, 2i} = \sin(\omega_i\, t), \qquad p_{t, 2i+1} = \cos(\omega_i\, t), \qquad i = 0, \dots, d/2 - 1 .
$$

The frequencies form a geometric progression from $\omega_0 = 1$ (period $2\pi$ positions) down to $\omega_{d/2-1} \approx 10^{-4}$ (period about 63,000 positions), so the encoding behaves like a binary counter with smoothly varying bits: fast dimensions distinguish neighbouring positions, slow ones distinguish distant regions.

The property that motivated this particular choice is that a **shift in position is a linear map that does not depend on the position**. Consider one frequency pair. By the angle-addition formulas,

$$
\begin{pmatrix} \sin\big(\omega (t+k)\big) \\ \cos\big(\omega (t+k)\big) \end{pmatrix}
=
\begin{pmatrix} \cos(\omega k) & \sin(\omega k) \\ -\sin(\omega k) & \cos(\omega k) \end{pmatrix}
\begin{pmatrix} \sin(\omega t) \\ \cos(\omega t) \end{pmatrix},
$$

a rotation by the angle $\omega k$ that depends on the offset $k$ only. Stacking these $2 \times 2$ rotations along the diagonal gives a matrix $M_k$ with $p_{t+k} = M_k\, p_t$ for *every* $t$. A query projection can therefore learn to ask "what is $k$ positions to my left?" as a fixed linear function, whatever the absolute position. A consequence is that the dot product $p_t^{\top} p_{t+k} = \sum_i \cos(\omega_i k)$ depends only on $k$ (it is symmetric in $k$ and largest at $k = 0$), which gives attention a built-in notion of relative distance. The cell implements the encoding, checks the shift property numerically, and plots both the encoding and the similarity profile.

In [ ]:
def sinusoidal_encoding(max_len, d_model, base=10000.0):
    """(max_len, d_model) table of fixed sinusoidal positional encodings."""
    pos = torch.arange(max_len, dtype=torch.float32)[:, None]                       # (T, 1)
    i = torch.arange(0, d_model, 2, dtype=torch.float32)                            # (d/2,)
    angle = pos / (base ** (i / d_model))                                           # (T, d/2): omega_i * t
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(angle)
    pe[:, 1::2] = torch.cos(angle)
    return pe


def shift_matrix(k, d_model, base=10000.0):
    """The block-diagonal rotation M_k with pe[t + k] = M_k @ pe[t] for all t."""
    omega = 1.0 / (base ** (torch.arange(0, d_model, 2, dtype=torch.float32) / d_model))
    M = torch.zeros(d_model, d_model)
    for j, w in enumerate(omega):
        c, s = math.cos(w.item() * k), math.sin(w.item() * k)
        M[2 * j:2 * j + 2, 2 * j:2 * j + 2] = torch.tensor([[c, s], [-s, c]])
    return M


T_PE, D_PE = 128, 64
pe = sinusoidal_encoding(T_PE, D_PE)
k = 7
M_k = shift_matrix(k, D_PE)
shifted = pe[:-k] @ M_k.T                                    # apply M_k to every row pe[t]
print(f"pe[t + {k}] == M_{k} pe[t] for all t:", torch.allclose(shifted, pe[k:], atol=1e-4))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].imshow(pe.cpu().numpy().T, aspect="auto", cmap="RdBu", origin="lower")
axes[0].set_xlabel("position t"); axes[0].set_ylabel("encoding dimension"); axes[0].set_title("Sinusoidal encoding"); axes[0].grid(False)
t0 = 40
sims = pe @ pe[t0]
axes[1].plot(np.arange(T_PE) - t0, sims.cpu().numpy())
axes[1].set_xlabel("offset k = t - 40"); axes[1].set_ylabel(r"$p_{40}^\top p_t$"); axes[1].set_title("Similarity depends on the offset only")
plt.tight_layout(); plt.show()

The left panel shows the counter-like structure: low dimensions oscillate fast along the position axis, high dimensions barely change over 128 positions. The right panel confirms that the similarity between position 40 and its neighbours is a symmetric function of the offset, peaking at 0.

### 4.3 Learned positional embeddings

GPT-2, BERT and many successors simply use a second embedding table, `nn.Embedding(max_len, d_model)`, indexed by position and trained with everything else. It is simpler, it performs about as well within the training length, and it has no built-in relative structure; its drawback is that positions beyond `max_len` do not exist, so the model cannot process longer sequences at all. Our GPT in Section 6 uses this design because it is the standard for small decoder-only models and it keeps the code short.

### 4.4 Rotary position embeddings (RoPE)

Sinusoidal and learned encodings are *added* to the token embedding once, at the input. Rotary embeddings (Su et al., 2021), used by most current large language models, instead act inside every attention layer, on the queries and keys only, by *rotating* each pair of coordinates $(q_{2i}, q_{2i+1})$ by an angle proportional to the position, $\theta_{i} t$ with the same geometric frequencies $\theta_i = 10000^{-2i/d}$ as above. Writing the rotation as $R_t$,

$$
\big(R_m q\big)^{\top} \big(R_n k\big) = q^{\top} R_m^{\top} R_n\, k = q^{\top} R_{n-m}\, k ,
$$

because rotations compose by adding angles. The attention score between a query at position $m$ and a key at position $n$ therefore depends on the *relative* offset $n - m$ and on the contents, never on the absolute positions. Values are left untouched, so the mixing of information is position-free while the decision of *where* to look is relative. The implementation is short: precompute cosines and sines per position and frequency, and apply the rotation to interleaved coordinate pairs.

In [ ]:
def rope_cos_sin(T, d, base=10000.0):
    """Cosine and sine tables of shape (T, d/2) for rotary embeddings."""
    inv_freq = 1.0 / (base ** (torch.arange(0, d, 2, dtype=torch.float32) / d))   # (d/2,)
    angles = torch.arange(T, dtype=torch.float32)[:, None] * inv_freq[None, :]     # (T, d/2)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    """Rotate coordinate pairs (x[2i], x[2i+1]) of x (..., T, d) by the position-dependent angles."""
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2] = x1 * cos - x2 * sin
    out[..., 1::2] = x1 * sin + x2 * cos
    return out


torch.manual_seed(SEED)
d_h = 16
cos, sin = rope_cos_sin(64, d_h)
q_vec, k_vec = torch.randn(d_h), torch.randn(d_h)

def rope_score(m, n):
    """Score between the same query content at position m and key content at position n."""
    qm = apply_rope(q_vec[None, :], cos[m:m + 1], sin[m:m + 1])[0]
    kn = apply_rope(k_vec[None, :], cos[n:n + 1], sin[n:n + 1])[0]
    return (qm @ kn).item()

print(f"score(m=5,  n=2 ) = {rope_score(5, 2):+.5f}")
print(f"score(m=13, n=10) = {rope_score(13, 10):+.5f}   <- same offset 3, same score")
print(f"score(m=40, n=37) = {rope_score(40, 37):+.5f}")
print(f"score(m=5,  n=3 ) = {rope_score(5, 3):+.5f}   <- offset 2: different")
print("norm is preserved by the rotation:", torch.allclose(apply_rope(q_vec[None], cos[9:10], sin[9:10]).norm(), q_vec.norm()))

The score is identical whenever the offset is identical, whatever the absolute positions, and the rotation preserves norms so it does not interfere with the $\sqrt{d_k}$ scaling. RoPE combines the advantages of the earlier schemes: relative structure like sinusoids, applied per layer, with no parameters. It also extrapolates to longer contexts somewhat better than learned embeddings, and a family of tricks (position interpolation, NTK-aware scaling, YaRN) extends pretrained models to longer contexts by rescaling its frequencies.

> **Key idea.** Attention alone is a set function. Positional encodings restore order: sinusoids and learned tables add position to the input; RoPE rotates queries and keys so that attention scores depend on relative offsets.

## 5. The Transformer block

### 5.1 Anatomy

A Transformer is a stack of identical **blocks**. Each block has two sub-layers, multi-head self-attention and a position-wise MLP, each wrapped in a **residual connection** and normalised with **LayerNorm**. In the *pre-LN* form that nearly all modern models use,

$$
\begin{aligned}
X' &= X + \operatorname{Dropout}\big(\operatorname{MultiHead}(\operatorname{LN}(X))\big), \\
X'' &= X' + \operatorname{Dropout}\big(\operatorname{MLP}(\operatorname{LN}(X'))\big), \qquad
\operatorname{MLP}(u) = W_2\, \operatorname{GELU}(W_1 u + b_1) + b_2 .
\end{aligned}
$$

Each piece has a job. **Attention** is the only place where positions exchange information; it *reads*. The **MLP**, applied to each position separately with the same weights (hence "position-wise"), *processes* what was read; it has a hidden width of $4d$, so two thirds of a block's parameters live here, and interpretability research suggests it behaves like a key–value memory of facts and patterns. **Residual connections** (Notebook 06 · Convolutional Neural Networks) give every block an identity path, so the network is a sum of small updates to a "residual stream" and gradients reach the embeddings undiminished. **LayerNorm** (Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization) normalises each token vector across its $d$ features, which keeps the scale of the residual stream stable across depth and, crucially, is independent of the batch and the sequence length, unlike BatchNorm.

### 5.2 Pre-LN versus post-LN

The 2017 paper placed LayerNorm *after* the residual addition ("post-LN"): $X' = \operatorname{LN}(X + \operatorname{MultiHead}(X))$. Post-LN Transformers are notoriously hard to train: they need a careful learning-rate warmup, and deep ones diverge without it. Xiong et al. (2020) showed why. In post-LN, the normalisation sits *on* the residual path, so the gradient with respect to the parameters of the last layers is large at initialisation and shrinks for earlier layers, giving an ill-conditioned problem; in pre-LN the residual path is a clean identity, the gradient norm is roughly uniform across layers, and training is stable even with a large initial learning rate. Pre-LN also needs a final LayerNorm before the output head, because the residual stream itself is never normalised. We use pre-LN throughout; `nn.TransformerEncoderLayer` exposes the choice as `norm_first=True`.

### 5.3 Encoder, decoder and decoder-only

The same block appears in three arrangements. An **encoder** stack (BERT) uses unmasked self-attention: every position sees the whole input, which is ideal for understanding tasks. A **decoder** stack (GPT) uses causal self-attention and is trained to predict the next token. The original **encoder–decoder** (translation, T5) adds a third sub-layer to each decoder block, cross-attention from the decoder to the encoder outputs. Section 9 returns to the three families; the block below is the causal one, written so that the attention module can optionally be *bidirectional* (for Exercise 4) and can optionally use a cache of past keys and values (Section 7). It uses `F.scaled_dot_product_attention` and applies attention dropout only in training mode.

In [ ]:
class SelfAttention(nn.Module):
    """Multi-head self-attention with a fused QKV projection, optional causal masking and a KV-cache."""

    def __init__(self, d_model, n_heads, dropout=0.0, causal=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_head, self.causal, self.dropout = n_heads, d_model // n_heads, causal, dropout
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x, cache=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)      # (B, h, T, d_h)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        p_drop = self.dropout if self.training else 0.0
        if cache is None:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal, dropout_p=p_drop)
        else:                                                            # incremental decoding (Section 7)
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
            T_q, T_k = q.shape[2], k.shape[2]
            mask = torch.ones(T_q, T_k, dtype=torch.bool, device=x.device).tril(diagonal=T_k - T_q)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=p_drop)
        y = y.transpose(1, 2).contiguous().view(B, T, C)                # merge the heads
        return self.proj(y)


class Block(nn.Module):
    """Pre-LN Transformer block: x + Attn(LN(x)), then x + MLP(LN(x))."""

    def __init__(self, d_model, n_heads, dropout=0.0, causal=True):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = SelfAttention(d_model, n_heads, dropout, causal)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(),
                                 nn.Linear(4 * d_model, d_model), nn.Dropout(dropout))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, cache=None):
        x = x + self.drop(self.attn(self.ln1(x), cache))
        x = x + self.mlp(self.ln2(x))
        return x


torch.manual_seed(SEED)
block = Block(d_model=64, n_heads=4, dropout=0.1)
x = torch.randn(2, 10, 64)
print("block output:", tuple(block(x).shape), "(same shape as the input: blocks can be stacked)")
block.eval()                                   # dropout off, so the comparison is deterministic
perm = torch.randperm(10)
with torch.no_grad():
    full = block(x)
    permuted = block(x[:, perm])
print("causal block is NOT permutation-equivariant (the mask depends on position):",
      not torch.allclose(permuted, full[:, perm], atol=1e-5))

### 5.4 Counting parameters and FLOPs

Being able to estimate the size and cost of a Transformer on the back of an envelope is a practical skill: it tells you whether a model fits in memory, how long training will take, and how the cost splits between attention and the MLP. Per block, with model width $d$ and biases included:

| Component | Parameters |
|---|---|
| attention projections $W_Q, W_K, W_V, W_O$ | $4d^2 + 4d$ |
| MLP $W_1 \in \mathbb{R}^{d \times 4d}$, $W_2 \in \mathbb{R}^{4d \times d}$ | $8d^2 + 5d$ |
| two LayerNorms (scale and shift) | $4d$ |
| **total per block** | $12d^2 + 13d \approx 12d^2$ |

A full decoder-only model with $L$ blocks, vocabulary $V$ and context $T_{\max}$ adds a token embedding ($Vd$, shared with the output head when tied), a positional table ($T_{\max} d$) and the final LayerNorm ($2d$): $N \approx 12 L d^2 + Vd + T_{\max} d$. For GPT-2 small ($L = 12$, $d = 768$, $V = 50257$, $T_{\max} = 1024$) this gives $85$M + $38.6$M + $0.8$M $\approx 124$M, the published figure.

For **FLOPs**, count a multiply–add as two operations. A matrix product of a $(T \times d_{\text{in}})$ activation with a $(d_{\text{in}} \times d_{\text{out}})$ weight costs $2 T d_{\text{in}} d_{\text{out}}$, i.e. $2 \times (\text{parameters})$ per token. Per token and per block the forward pass therefore costs $2 \times 12 d^2 = 24 d^2$ for the weight matrices, plus the two attention products $QK^{\top}$ and $AV$, which each cost $2 T d$ per token (a $d$-dimensional dot product with each of $T$ keys, summed over heads): $4Td$ in total. Backpropagation costs about twice the forward pass, so training is $\approx 3\times$ forward:

$$
\text{FLOPs per token (forward)} \approx L\,(24 d^2 + 4 T d) + 2Vd, \qquad
\text{FLOPs per token (training)} \approx 6N + 12 L T d .
$$

The famous **$6N$ rule** (training FLOPs $\approx 6 \times \text{parameters} \times \text{tokens}$) is exactly this formula with the attention term dropped, which is accurate as long as $T \ll 6d$; for long contexts the $12 L T d$ term takes over, and that is the quadratic cost of attention in disguise. The cell checks the parameter formula against the real module and tabulates the FLOP split for several configurations.

In [ ]:
def block_params_formula(d):
    return 12 * d * d + 13 * d

def gpt_params_formula(L, d, V, T_max):
    return L * block_params_formula(d) + V * d + T_max * d + 2 * d

def flops_per_token_forward(L, d, V, T):
    return L * (24 * d * d + 4 * T * d) + 2 * V * d

d_check = 64
print(f"block(d={d_check}) parameters: actual {sum(p.numel() for p in block.parameters()):,} "
      f"| formula 12d^2 + 13d = {block_params_formula(d_check):,}")
print(f"GPT-2 small parameters by formula: {gpt_params_formula(12, 768, 50257, 1024) / 1e6:.1f} M (published: 124 M)\n")

configs = [("notebook GPT (FAST)", 2, 64, 65, 64), ("notebook GPT (full)", 6, 256, 65, 256),
           ("GPT-2 small", 12, 768, 50257, 1024), ("GPT-2 XL", 48, 1600, 50257, 1024),
           ("GPT-3 175B", 96, 12288, 50257, 2048)]
print(f"{'model':>20} | {'params':>9} | {'fwd MFLOPs/token':>16} | {'attention share':>15} | {'6N (MFLOPs)':>12}")
for name, L, d, V, T in configs:
    N = gpt_params_formula(L, d, V, T)
    fwd = flops_per_token_forward(L, d, V, T)
    attn_share = L * 4 * T * d / fwd
    print(f"{name:>20} | {N / 1e6:8.1f}M | {fwd / 1e6:16,.1f} | {attn_share:14.1%} | {6 * N / 1e6:12,.0f}")

The formula reproduces the module's parameter count exactly and GPT-2's published size within rounding. At the contexts these models were trained with, the attention products are a modest share of the forward FLOPs, roughly 9–13% for GPT-2-sized models at 1,024 tokens and a few percent for GPT-3 (whose $d$ is much larger), which is why the $6N$ rule is a good approximation for them; Section 8 shows what happens when the context grows to hundreds of thousands of tokens.

> **Common pitfall.** Parameter count and *memory* are different things. Training with Adam needs the parameters, their gradients and two moment estimates (four copies in float32, 16 bytes per parameter) plus the activations of every layer for the backward pass, which scale with batch size × sequence length × depth × width. Notebook 14 · Efficient Training and Scaling does the full memory arithmetic.

## 6. Build a GPT

### 6.1 Data and tokenizer

We now assemble the pieces into a complete decoder-only language model in the style of GPT-2 (Radford et al., 2019) and train it from scratch on Tiny Shakespeare, the same character-level corpus as in Notebook 08 · Sequence Models: RNNs, LSTMs and GRUs, so that the two model families can be compared on equal terms. The download is guarded and falls back to a short built-in text when offline; the tokenizer is again the sorted set of characters. One difference from the recurrent models of that notebook: a Transformer has no state to carry between chunks, so we do not need contiguous streams. Each training example is simply a random window of `block_size` characters and its one-step-shifted copy, and the model predicts *every* next character in the window in parallel, thanks to the causal mask.

In [ ]:
FALLBACK_TEXT = """FIRST CITIZEN:
Before we go one step further, hear me speak.
The granaries are full, and yet we starve;
The gates are shut against us, and the guard
That should be ours is paid to keep us out.

SECOND CITIZEN:
Speak on, for hunger makes a patient listener.

FIRST CITIZEN:
Then hear this: he that holds the keys tonight
Holds all our children's suppers in his fist.
Shall we go begging to a bolted door,
Or shall we make the door remember us?

ALL:
Let it remember! To the gates, to the gates!

SECOND CITIZEN:
Soft, soft. A crowd is brave until it meets
A single man who will not step aside.
What would you have us do when we are there?

FIRST CITIZEN:
Ask, first. Then ask again with louder voices.
And if the third request should find no ear,
The fourth shall need no words.

Enter a STEWARD, with torches.

STEWARD:
Good people, what is this? The hour is late,
The lord is sleeping, and the streets are cold.
Go home and warm yourselves; the morning brings
A fairer time for grievances than night.

FIRST CITIZEN:
The morning brings the same shut gate, good steward,
And one more empty bowl upon the table.
We are not here to quarrel with the dark;
We are here because the dark has come indoors.

STEWARD:
I am a servant, not the master of the store.
What I am told to keep, I keep; what I am told
To open, that I open. Bring me his command
And I will fling the doors so wide
The very rats will think the harvest has come twice.

SECOND CITIZEN:
Then fetch him, and be quick; we shall be counting.

STEWARD:
He will not come for counting. He will come
For fire, or not at all.

FIRST CITIZEN:
You heard him, friends. He asks us for a fire.

SECOND CITIZEN:
He asked for no such thing; he named the cost
Of being heard, and hoped we could not pay it.

FIRST CITIZEN:
Then let us pay in coin he did not name.
Sit down, all of you. Sit down upon the stones
Before his door, and let no wagon pass,
No cart, no horse, no message and no bread,
Until the keys come down. We need no torches;
Our patience is the only flame we bring,
And it will burn the longer for being cold.

ALL:
Sit! Sit! Let no wagon pass!

LORD HALDEN:
What noise is this, that will not be a noise?
I looked for shouting, and I find a silence
That has more teeth in it than any shout.
Steward! What do they want?

STEWARD:
They want the gates, my lord, and they will wait.

LORD HALDEN:
Wait? Let them wait; the night is theirs to waste.

STEWARD:
The night is theirs, my lord, and so is the road.
Nothing has come in since they sat down,
And nothing has gone out. Your cousin's grain
Stands in three carts a mile beyond the bridge
And cannot pass. The buyer from the coast
Has turned his mule around.

LORD HALDEN:
So. They have found the one thing that I fear,
Which is not fire, but the cost of stillness.
Well then. A man who counts must sometimes count
What he would rather not. Give them the keys,
But say it was my mercy, not their sitting.

STEWARD:
They will not care what it is called, my lord,
So long as it is bread.

LORD HALDEN:
No. And that is why they will be back.

FIRST CITIZEN:
The keys! The keys come down! Rise up, and gently;
We came as neighbours, and we leave as such.
Take what you need and mark what you have taken,
For we will pay it back when harvest comes,
And show him that a debt can be a bridge.

SECOND CITIZEN:
And if he burns the bridge?

FIRST CITIZEN:
Then we have learned to sit, and can sit twice.

STEWARD:
I have served twenty years behind that gate
And never once was asked what I would open.
Tonight a crowd of strangers asked me nothing,
And sat, and by their sitting turned the key
That all my service never dared to touch.
What is a lock, if not a kind of promise
That someone on the other side will listen?
And what is service, if the promise breaks?
I think I shall go home by the long road,
And count the carts, and count the empty bowls,
And learn which of the two I am afraid of.
"""


def load_tiny_shakespeare(data_dir, min_chars=100_000):
    """Download Tiny Shakespeare into data_dir once; fall back to the built-in text when offline."""
    path = data_dir / "tinyshakespeare.txt"
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    if not path.exists():
        try:
            with urllib.request.urlopen(url, timeout=30) as response:
                raw = response.read().decode("utf-8")
            path.write_text(raw, encoding="utf-8")
            print(f"Downloaded Tiny Shakespeare to {path} ({len(raw):,} characters).")
        except Exception as exc:
            print(f"Could not download Tiny Shakespeare ({type(exc).__name__}: {exc}).")
            print("Using the built-in fallback text instead, so that the notebook still runs offline.")
    if path.exists() and path.stat().st_size >= min_chars:
        return path.read_text(encoding="utf-8"), "Tiny Shakespeare"
    return FALLBACK_TEXT, "built-in fallback text"


text, text_source = load_tiny_shakespeare(DATA_DIR)
chars = sorted(set(text))
VOCAB_SIZE = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[int(i)] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n_split = int(0.9 * len(data))
train_data, val_data = data[:n_split], data[n_split:]
print(f"corpus: {text_source} | {len(text):,} characters | vocabulary {VOCAB_SIZE} "
      f"| train {len(train_data):,} | val {len(val_data):,}")


def get_batch(data, batch_size, block_size, generator=None):
    """Random windows of block_size tokens and their one-step-shifted targets, moved to the device."""
    ix = torch.randint(len(data) - block_size - 1, (batch_size,), generator=generator)
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


xb, yb = get_batch(train_data, 4, 16)
print("batch x:", tuple(xb.shape), "| y:", tuple(yb.shape))
print("x[0]:", repr(decode(xb[0].tolist())), "-> y[0]:", repr(decode(yb[0].tolist())))

### 6.2 The `GPT` module

The model is the sum of a token embedding and a learned positional embedding, followed by `n_layers` causal blocks, a final LayerNorm (required by pre-LN, Section 5.2) and a linear **language-model head** that maps each position's $d$-dimensional state to `VOCAB_SIZE` logits. Two details are standard practice and worth understanding.

**Weight tying** (Press and Wolf, 2017). The input embedding maps a token to a vector; the output head maps a vector to a score per token. Using the *same* matrix for both, $\text{logits} = h\, E^{\top}$, halves the embedding parameters and acts as a useful inductive bias: the score for token $w$ is the dot product between the state and $w$'s embedding, so tokens with similar embeddings get similar scores. In PyTorch it is one assignment, `self.lm_head.weight = self.tok_emb.weight`, and the shared tensor appears once in `parameters()`. For character models the saving is small; for a 50k-token vocabulary it is tens of millions of parameters.

**Initialisation.** Weights are drawn from $\mathcal{N}(0, 0.02^2)$ as in GPT-2, and the two projections that write *into* the residual stream (the attention output projection and the second MLP layer) are scaled down by $1/\sqrt{2L}$, because $2L$ such writes are summed along the residual path and their variance would otherwise grow with depth.

The `forward` method returns logits and, when targets are given, the mean cross-entropy over all positions; the `caches` and `start_pos` arguments are used only by the incremental decoder of Section 7.

In [ ]:
class GPT(nn.Module):
    """Decoder-only Transformer language model (GPT-2 style, pre-LN, tied embeddings)."""

    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_len, dropout=0.0):
        super().__init__()
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, dropout, causal=True) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight                    # weight tying
        self.apply(self._init_weights)
        for name, p in self.named_parameters():                      # scaled init of residual writes
            if name.endswith("attn.proj.weight") or name.endswith("mlp.2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layers))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, caches=None, start_pos=0):
        B, T = idx.shape
        assert start_pos + T <= self.max_len, "sequence longer than the model's context"
        pos = torch.arange(start_pos, start_pos + T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))          # (B, T, d)
        for i, block in enumerate(self.blocks):
            x = block(x, None if caches is None else caches[i])
        logits = self.lm_head(self.ln_f(x))                           # (B, T, vocab)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


if FAST_RUN:
    cfg = dict(n_layers=2, d_model=64, n_heads=4, block_size=64, batch_size=32, steps=500,
               lr=3e-3, min_lr=3e-4, warmup=50, weight_decay=0.1, dropout=0.0, eval_every=100, eval_batches=10)
else:
    cfg = dict(n_layers=6, d_model=256, n_heads=8, block_size=256, batch_size=64, steps=4000,
               lr=6e-4, min_lr=6e-5, warmup=200, weight_decay=0.1, dropout=0.1, eval_every=250, eval_batches=20)

torch.manual_seed(SEED)
gpt = GPT(VOCAB_SIZE, cfg["d_model"], cfg["n_heads"], cfg["n_layers"], cfg["block_size"], cfg["dropout"]).to(device)
n_params = sum(p.numel() for p in gpt.parameters())
print(f"GPT: {cfg['n_layers']} layers, d_model {cfg['d_model']}, {cfg['n_heads']} heads, context {cfg['block_size']} "
      f"-> {n_params:,} parameters (formula: {gpt_params_formula(cfg['n_layers'], cfg['d_model'], VOCAB_SIZE, cfg['block_size']):,})")
print("lm_head shares its weight with tok_emb:", gpt.lm_head.weight.data_ptr() == gpt.tok_emb.weight.data_ptr())
xb, yb = get_batch(train_data, 2, cfg["block_size"])
logits, loss = gpt(xb, yb)
print(f"logits {tuple(logits.shape)} | initial loss {loss.item():.3f} (uniform guess would be ln {VOCAB_SIZE} = {math.log(VOCAB_SIZE):.3f})")

The initial loss is close to $\ln V$, which confirms that the output distribution starts out nearly uniform; a much larger value would indicate an initialisation that produces confident wrong predictions.

### 6.3 The training recipe

The recipe below is the one used, with different numbers, by essentially every Transformer language model since GPT-2.

- **AdamW** (Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization) with $\beta_2 = 0.95$ rather than the default $0.999$, which reacts faster to changes in gradient scale and is standard for Transformers, and **decoupled weight decay** of 0.1 applied to the weight matrices only. Biases and LayerNorm gains are conventionally excluded from decay: we decay every parameter with two or more dimensions and exempt the rest. This simple rule also decays the embedding tables (as nanoGPT does); some codebases exempt them as well, and the difference is minor.
- **Linear warmup followed by cosine decay** of the learning rate. Adam's second-moment estimate is unreliable during the first steps, and a full-size learning rate applied to a freshly initialised Transformer produces large, destabilising updates; warming up over a few hundred steps avoids this. Cosine decay to about a tenth of the peak then lets the loss settle at the end.
- **Gradient clipping** at norm 1.0, as in Notebook 08 · Sequence Models: RNNs, LSTMs and GRUs; we log the pre-clipping norm, which is a useful health indicator (spikes precede divergence).
- **Evaluation** on a fixed set of random validation windows (same windows every time, via a seeded generator), and a short **sample** at a few checkpoints.

In [ ]:
def lr_at(step, max_lr, warmup, total_steps, min_lr):
    """Linear warmup to max_lr over `warmup` steps, then cosine decay to min_lr at total_steps."""
    if step < warmup:
        return max_lr * (step + 1) / warmup
    progress = min(1.0, (step - warmup) / max(1, total_steps - warmup))
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


def configure_optimizer(model, lr, weight_decay):
    """AdamW with weight decay on every parameter of dim >= 2 (matrices and embedding tables); none on biases and LayerNorm gains."""
    decay = [p for p in model.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.dim() < 2]
    groups = [{"params": decay, "weight_decay": weight_decay}, {"params": no_decay, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=lr, betas=(0.9, 0.95))


@torch.no_grad()
def estimate_loss(model, data, batch_size, block_size, n_batches):
    """Mean loss over a fixed set of random windows (the same windows at every call)."""
    was_training = model.training
    model.eval()
    g = torch.Generator().manual_seed(SEED)
    losses = [model(*get_batch(data, batch_size, block_size, generator=g))[1].item() for _ in range(n_batches)]
    if was_training:
        model.train()
    return float(np.mean(losses))


@torch.no_grad()
def sample_simple(model, idx, max_new_tokens, temperature=1.0):
    """Plain temperature sampling, used for checkpoint samples (all decoding options come in Section 7)."""
    was_training = model.training
    model.eval()
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -model.max_len:])
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, num_samples=1)], dim=1)
    if was_training:
        model.train()
    return idx


steps_axis = np.arange(cfg["steps"])
fig, ax = plt.subplots()
ax.plot(steps_axis, [lr_at(s, cfg["lr"], cfg["warmup"], cfg["steps"], cfg["min_lr"]) for s in steps_axis])
ax.set_xlabel("step"); ax.set_ylabel("learning rate"); ax.set_title("Warmup + cosine schedule")
plt.tight_layout(); plt.show()

The training loop itself is short. Under `FAST_RUN` (2 layers, $d = 64$, context 64, 500 steps of 32×64 characters) it takes under a minute on a CPU and reaches a validation loss around 2.2 nats per character; the full run (6 layers, $d = 256$, context 256, 4,000 steps, about 5M parameters) takes a few minutes on a GPU and reaches around 1.4–1.5 nats per character, better than the LSTM of Notebook 08 · Sequence Models: RNNs, LSTMs and GRUs at a comparable parameter count.

In [ ]:
def train_gpt(model, train_data, val_data, cfg, prompt_idx, sample_steps):
    opt = configure_optimizer(model, cfg["lr"], cfg["weight_decay"])
    history = {"step": [], "train": [], "val": []}
    samples = {}
    model.train()
    running, t0 = None, time.time()
    for step in range(1, cfg["steps"] + 1):
        lr = lr_at(step - 1, cfg["lr"], cfg["warmup"], cfg["steps"], cfg["min_lr"])
        for group in opt.param_groups:
            group["lr"] = lr
        x, y = get_batch(train_data, cfg["batch_size"], cfg["block_size"])
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        running = loss.item() if running is None else 0.9 * running + 0.1 * loss.item()
        if step % cfg["eval_every"] == 0 or step == cfg["steps"]:
            val = estimate_loss(model, val_data, cfg["batch_size"], cfg["block_size"], cfg["eval_batches"])
            history["step"].append(step); history["train"].append(running); history["val"].append(val)
            print(f"step {step:5d} | lr {lr:.2e} | train {running:.3f} | val {val:.3f} "
                  f"| grad norm {grad_norm:.2f} | {time.time() - t0:6.1f}s")
        if step in sample_steps:
            samples[step] = decode(sample_simple(model, prompt_idx, 200, temperature=0.8)[0].tolist())
    return history, samples


PROMPT = "ROMEO:\n" if all(c in stoi for c in "ROMEO:\n") else text[:8]
prompt_idx = torch.tensor([encode(PROMPT)], dtype=torch.long, device=device)
SAMPLE_STEPS = sorted({1, cfg["steps"] // 4, cfg["steps"] // 2, cfg["steps"]})
print(f"training for {cfg['steps']} steps of {cfg['batch_size']}x{cfg['block_size']} = "
      f"{cfg['batch_size'] * cfg['block_size']:,} tokens each ({cfg['steps'] * cfg['batch_size'] * cfg['block_size'] / 1e6:.1f}M tokens)")
gpt_history, gpt_samples = train_gpt(gpt, train_data, val_data, cfg, prompt_idx, SAMPLE_STEPS)

The gradient norm is typically above 1 during the first steps (clipping is active) and settles below the threshold as the loss stabilises. The loss curve of a Transformer language model has a characteristic shape: a very fast initial drop as unigram and bigram statistics are absorbed, then a long, slow, roughly log-linear improvement as the model discovers progressively longer-range structure. Let us plot it and read the checkpoint samples.

In [ ]:
fig, ax = plt.subplots()
ax.plot(gpt_history["step"], gpt_history["train"], marker="o", label="train (moving average)")
ax.plot(gpt_history["step"], gpt_history["val"], marker="s", label="validation")
ax.set_xlabel("step"); ax.set_ylabel("cross-entropy (nats per character)")
ax.set_title(f"GPT on {text_source}: {n_params / 1e6:.2f}M parameters")
ax.legend(); plt.tight_layout(); plt.show()
print(f"final validation loss {gpt_history['val'][-1]:.3f} nats -> perplexity {math.exp(gpt_history['val'][-1]):.2f} per character\n")
for step, s in gpt_samples.items():
    print(f"----- sample after step {step} -----")
    print(s)
    print()

The progression mirrors what we saw with the LSTM: noise, then the shape of a play script (short lines, capitalised speaker names and colons), then word-like fragments, then real words and phrases. Under `FAST_RUN` the model is tiny (two layers of width 64, about 100k parameters) and has seen only a million characters, so expect it to reach the "fragments and short words" stage; the full run gets to phrases. There the attention mechanism shows its strength: the model learns to copy a speaker's name from earlier in the context, to close a quotation it opened, and to keep a consistent register within a speech, all of which require reading specific earlier positions rather than a compressed summary. At this scale the model still has no understanding of what it writes; what changes with scale is not the recipe but the data, the parameter count and the compute (Section 8).

> **Try it.** Set `cfg["dropout"] = 0.2` under `FAST_RUN` and retrain: with a 500-step budget on a million characters, dropout only slows learning, because the model is nowhere near overfitting. Exercise 2 sets up the regime where it helps.

## 7. Generation and the KV-cache

### 7.1 Decoding strategies

A trained language model gives us $p(x_t \mid x_{<t})$; turning that into text requires a **decoding strategy**, and the choice matters as much as the model for the perceived quality of the output. All strategies below operate on the logits $z$ of the last position.

- **Greedy decoding** takes $\arg\max_i z_i$ at every step. It is deterministic and locally optimal, and it tends to produce repetitive, looping text, because the most likely continuation of a phrase is often the phrase again.
- **Temperature sampling** draws from $\operatorname{softmax}(z / \tau)$, as in Notebook 08 · Sequence Models: RNNs, LSTMs and GRUs; $\tau < 1$ sharpens, $\tau > 1$ flattens.
- **Top-$k$ sampling** (Fan et al., 2018) keeps only the $k$ highest logits, sets the rest to $-\infty$, and samples from the renormalised distribution. It removes the "unreliable tail", the thousands of tokens that individually have tiny probability but together hold a lot of mass, which is where sampling at $\tau = 1$ goes wrong.
- **Top-$p$ (nucleus) sampling** (Holtzman et al., 2020) keeps the smallest set of tokens whose cumulative probability reaches $p$: the set is large when the model is uncertain and small when it is confident, unlike a fixed $k$. The implementation sorts the probabilities, takes the cumulative sum, and removes every token whose cumulative mass *before* it already exceeds $p$; this always keeps at least one token and always includes the token that crosses the threshold (Exercise 3 tests the edge cases).
- **Repetition penalty** (Keskar et al., 2019) divides the logits of tokens that already appear in the context by a factor $\theta > 1$ (and multiplies negative logits by $\theta$), discouraging loops at the cost of sometimes suppressing legitimately repeated tokens.

The `generate` function combines all of these; each call re-seeds so that the samples differ only in the strategy.

In [ ]:
def top_k_filter(logits, k):
    """Keep the k largest logits per row; set the rest to -inf."""
    k = min(k, logits.shape[-1])
    threshold = torch.topk(logits, k, dim=-1).values[..., -1:]
    return logits.masked_fill(logits < threshold, float("-inf"))


def top_p_filter(logits, p):
    """Nucleus filtering: keep the smallest set of tokens whose probability mass reaches p."""
    sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
    probs = F.softmax(sorted_logits, dim=-1)
    cumulative = probs.cumsum(dim=-1)
    remove = (cumulative - probs) > p                     # mass *before* this token already exceeds p
    sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
    return torch.full_like(logits, float("-inf")).scatter(-1, sorted_idx, sorted_logits)


@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None,
             repetition_penalty=1.0, greedy=False):
    """Autoregressive decoding with greedy / temperature / top-k / top-p / repetition-penalty options."""
    was_training = model.training
    model.eval()
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -model.max_len:])
        logits = logits[:, -1, :].clone()
        if repetition_penalty != 1.0:
            for b in range(idx.shape[0]):
                seen = idx[b].unique()
                z = logits[b, seen]
                logits[b, seen] = torch.where(z > 0, z / repetition_penalty, z * repetition_penalty)
        if greedy:
            nxt = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k is not None:
                logits = top_k_filter(logits, top_k)
            if top_p is not None:
                logits = top_p_filter(logits, top_p)
            nxt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
        idx = torch.cat([idx, nxt], dim=1)
    if was_training:
        model.train()
    return idx


N_GEN = 200
strategies = {
    "greedy": dict(greedy=True),
    "temperature 0.8": dict(temperature=0.8),
    "temperature 1.0": dict(temperature=1.0),
    "top-k 10, temperature 1.0": dict(top_k=10),
    "top-p 0.9, temperature 1.0": dict(top_p=0.9),
    "greedy + repetition penalty 1.3": dict(greedy=True, repetition_penalty=1.3),
}
for name, kwargs in strategies.items():
    torch.manual_seed(SEED)
    out = generate(gpt, prompt_idx, N_GEN, **kwargs)
    print(f"===== {name} =====")
    print(decode(out[0].tolist()))
    print()

Compare the samples. Greedy decoding is fluent for a few words at most and then locks into a loop (with the tiny model typically "the the the ..."), the classic failure of likelihood-maximising decoding. The repetition penalty is designed for word or subword vocabularies, where a repeated *token* is a repeated word; with a character vocabulary nearly every token has already appeared after a few dozen characters, so the penalty applies almost uniformly and does little, as the last sample shows. Temperature 1 is diverse and error-prone; 0.8 is the usual compromise. Top-$k$ and top-$p$ keep the diversity of $\tau = 1$ while cutting the tail, which is why some form of truncated sampling is the default in every deployed text generator. With a character-level model the tail is only a few dozen characters wide, so the effect is subtler than with a 50k-token vocabulary, but it is visible in the frequency of misspellings and stray punctuation.

### 7.2 Why naive generation is quadratic, and the KV-cache

Look at what `generate` does at each step: it runs the *whole* context through the model to obtain the logits of the last position only. Generating $n$ tokens after a prompt of length $T_0$ therefore processes $\sum_{i=0}^{n-1} (T_0 + i) = O(n T_0 + n^2 / 2)$ token-positions, and most of that work is repeated: the keys and values that position $j$ offers in every layer depend only on tokens $1, \dots, j$ (causal masking), so they are *identical* at every later step. The **KV-cache** stores them. After the prompt is processed once ("prefill"), each new token is fed *alone*: its query attends to the cached keys and values plus its own, the new key and value are appended to the cache, and the cost per step becomes $O(T)$ for attention plus a constant for the MLP, instead of a full forward pass over $T$ tokens.

Our `SelfAttention` already supports this through the `cache` argument (one dictionary per layer holding `k` and `v` of shape `(B, h, T_so_far, d_h)`), and `GPT.forward` accepts `start_pos` so that the new token receives the correct positional embedding. The mask in the cached path is built explicitly, `tril(diagonal=T_k - T_q)`, which equals the ordinary causal mask during prefill ($T_q = T_k$) and allows the single new query to see everything when $T_q = 1$. The cell implements cached generation, checks that greedy decoding produces the same tokens with and without the cache, and times both.

In [ ]:
@torch.no_grad()
def generate_cached(model, idx, max_new_tokens, temperature=1.0, greedy=False):
    """Incremental decoding with a per-layer KV-cache (prompt + generated tokens must fit the context)."""
    was_training = model.training
    model.eval()
    T0 = idx.shape[1]
    assert T0 + max_new_tokens <= model.max_len, "this simple cache does not slide the context window"
    caches = [dict() for _ in model.blocks]
    logits, _ = model(idx, caches=caches, start_pos=0)                 # prefill: process the prompt once
    tokens = [idx]
    for i in range(max_new_tokens):
        last = logits[:, -1, :]
        if greedy:
            nxt = last.argmax(dim=-1, keepdim=True)
        else:
            nxt = torch.multinomial(F.softmax(last / temperature, dim=-1), num_samples=1)
        tokens.append(nxt)
        if i + 1 < max_new_tokens:
            logits, _ = model(nxt, caches=caches, start_pos=T0 + i)   # only the new token goes through
    if was_training:
        model.train()
    return torch.cat(tokens, dim=1)


n_new = cfg["block_size"] - prompt_idx.shape[1] - 1
out_plain = generate(gpt, prompt_idx, n_new, greedy=True)
out_cached = generate_cached(gpt, prompt_idx, n_new, greedy=True)
agree = (out_plain == out_cached).float().mean().item()
print(f"greedy tokens identical with and without cache: {agree:.1%} of {out_plain.shape[1]} positions")

t_plain = time_fn(lambda: generate(gpt, prompt_idx, n_new, greedy=True), n_repeat=3)
t_cached = time_fn(lambda: generate_cached(gpt, prompt_idx, n_new, greedy=True), n_repeat=3)
T0 = prompt_idx.shape[1]
positions_plain = sum(T0 + i for i in range(n_new))
positions_cached = T0 + n_new - 1
cache_bytes = 2 * cfg["n_layers"] * cfg["n_heads"] * (T0 + n_new) * (cfg["d_model"] // cfg["n_heads"]) * 4
print(f"generating {n_new} tokens: without cache {t_plain * 1e3:7.1f} ms | with cache {t_cached * 1e3:7.1f} ms "
      f"| speed-up {t_plain / t_cached:.1f}x")
print(f"token-positions processed: {positions_plain:,} without cache vs {positions_cached:,} with cache "
      f"({positions_plain / positions_cached:.1f}x fewer) | cache size {cache_bytes / 1024:.0f} KiB (float32)")

The two decoders agree token for token (any disagreement would come from near-ties in the logits resolved differently by the two numerically slightly different code paths). The measured speed-up is smaller than the reduction in token-positions because, for a tiny model on a CPU, every step pays a fixed overhead of kernel launches and Python that the cache does not remove; for a real model on a GPU, where the per-token compute dominates, the speed-up approaches the ratio of positions and grows with the generation length. The price is memory: the cache holds $2 \times L \times T \times d$ values per sequence, which for a 70B-parameter model ($L = 80$, $d = 8192$) with standard multi-head attention at a context of 8k tokens is about 20 GiB per sequence in float16, on top of the 140 GB of weights. That memory is the reason for the multi-query and grouped-query attention variants of Exercise 5, and for the paged-attention schemes used by inference servers.

**Beam search**, common in translation and summarisation, keeps the $B$ most probable partial sequences at every step instead of one, expanding each by the vocabulary and pruning back to $B$ by total log-probability (usually length-normalised). It finds higher-likelihood outputs than greedy decoding, which is what a translation should be, but for open-ended generation "most likely" text is bland and repetitive, which is why sampling-based strategies dominate there (Holtzman et al., 2020). Each beam needs its own KV-cache, so the memory cost scales with $B$.

> **Key idea.** Decoding turns a next-token distribution into text: truncated sampling (top-$k$ / top-$p$) with a moderate temperature is the practical default. The KV-cache makes autoregressive generation linear in the context length by reusing the keys and values that causal attention guarantees never change.

## 8. Scaling

### 8.1 Parameters, data and compute

The recipe of Section 6 is, up to constants, the recipe of the largest language models. What separates our five-million-parameter toy from them is scale, and the striking empirical fact of the last few years is that scale behaves *predictably*. Kaplan et al. (2020) measured the test loss of Transformer language models over several orders of magnitude of model size $N$, dataset size $D$ (in tokens) and training compute $C$, and found smooth **power laws**: when none of the other two quantities is the bottleneck, the loss falls as $L(N) \approx (N_c / N)^{\alpha_N}$ with $\alpha_N \approx 0.076$, and similarly for $D$ and $C$. Doubling the model does not double the quality, it shaves a fixed *fraction* off the loss (off the excess loss above an irreducible floor, in the refined form of the law), and the same law that describes a 10M model describes a 10B one. Architecture details (depth versus width, number of heads) matter far less than the total parameter count, as long as they stay within reasonable ranges.

The laws answer a practical question: given a fixed compute budget $C \approx 6ND$ (Section 5.4), how should it be split between a bigger model and more tokens? Hoffmann et al. (2022), the "Chinchilla" paper, trained over 400 models to settle it and found that $N$ and $D$ should be scaled *together*, in roughly equal proportion; their compute-optimal frontier corresponds to about **20 tokens per parameter**. Earlier models such as GPT-3 (175B parameters, 300B tokens, 1.7 tokens per parameter) and Gopher (280B parameters, 300B tokens) were, by this measure, far too large for their data; Chinchilla (70B parameters, 1.4T tokens), trained with the same compute as Gopher, outperformed both with a quarter of Gopher's parameters. The rule of thumb has since shifted further towards *more* tokens per parameter (Llama-family models use several hundred to a couple of thousand), not because the compute-optimal frontier moved but because inference cost, which scales with $N$ and not with $D$, favours smaller models trained for longer than compute-optimality alone would suggest.

The cell applies the two formulas to a few well-known configurations: parameter count, Chinchilla-optimal token count, the resulting training FLOPs, and how long that takes on an accelerator delivering one petaflop per second at a realistic 30% utilisation.

In [ ]:
print(f"{'model':>14} | {'params':>9} | {'tokens (20/param)':>17} | {'train FLOPs':>11} | {'1 PFLOP/s @30%':>15}")
for name, L, d, V, T in [("notebook full", 6, 256, 65, 256), ("GPT-2 small", 12, 768, 50257, 1024),
                         ("GPT-2 XL", 48, 1600, 50257, 1024), ("GPT-3", 96, 12288, 50257, 2048),
                         ("Chinchilla-70B", 80, 8192, 32000, 2048)]:
    N = gpt_params_formula(L, d, V, T)
    D_opt = 20 * N
    C = 6 * N * D_opt
    days = C / (1e15 * 0.3) / 86400
    print(f"{name:>14} | {N / 1e9:8.3f}B | {D_opt / 1e9:15.1f}B | {C:11.2e} | {days:10.1f} days")

The table makes the exponential cost of scale concrete: each ten-fold increase in parameters, trained compute-optimally, costs a hundred-fold more compute, because the tokens grow with the parameters. It also shows why the field became obsessed with efficiency: a model that is 3× cheaper to train at equal quality is worth more than most architectural novelties.

### 8.2 Why longer context is quadratic

Attention compares every query with every key, so the score matrix has $T^2$ entries per head and per layer, and the FLOPs of the attention products grow as $4Td$ per token, i.e. $4T^2 d$ per sequence and layer (Section 5.4). At the contexts of 2017–2020 (512–2,048 tokens) this term was a small fraction (roughly 3–13%, Section 5.4) of the $24d^2$ of the projections and the MLP. At 128k tokens it dominates. Memory is worse: the from-scratch implementation stores the $T \times T$ matrix per head, which is why FlashAttention's linear memory (Section 3.4) was a precondition for long contexts. The KV-cache at inference is linear in $T$ but with a large constant, $2Ld$ values per token.

Many **efficient attention** variants trade exactness for cost: *sparse* patterns (Longformer, BigBird: sliding windows plus a few global tokens), *low-rank* approximations of the softmax kernel (Linformer, Performer), *recurrent compression* of the past (Transformer-XL, Compressive Transformer), and *state-space models* (Mamba) that replace attention with a linear recurrence. In practice, exact attention with FlashAttention, RoPE and multi-query or grouped-query heads (Exercise 5) has remained the default for the largest models, with sliding-window attention in some layers as the most widely adopted approximation. The cell quantifies the shift for a GPT-2-small-sized model as its context grows.

In [ ]:
L_, d_, V_ = 12, 768, 50257
print(f"GPT-2 small (L={L_}, d={d_}) as the context T grows:")
print(f"{'T':>8} | {'attention share of fwd FLOPs':>28} | {'score matrices / layer (fp16)':>29} | {'KV-cache / sequence (fp16)':>26}")
for T in [1024, 4096, 16384, 65536, 131072]:
    fwd = flops_per_token_forward(L_, d_, V_, T)
    share = L_ * 4 * T * d_ / fwd
    scores_bytes = 12 * T * T * 2                         # 12 heads, float16
    kv_bytes = 2 * L_ * T * d_ * 2
    print(f"{T:8d} | {share:27.1%} | {scores_bytes / 2**30:26.2f} GiB | {kv_bytes / 2**30:23.2f} GiB")

At 1k tokens attention is a modest share of the FLOP count; at 128k it is nearly all of the computation, the naive score matrices would need hundreds of gigabytes per layer, and even the KV-cache of this small model reaches several gigabytes per sequence. Long-context models are therefore less about the Transformer's mathematics than about its engineering.

## 9. The Transformer family

### 9.1 Three arrangements, three objectives

**Encoder-only (BERT, Devlin et al., 2019).** A stack of *unmasked* blocks reads the whole input at once; every position sees every other. Such a model cannot generate text (it would see its own answer), so it is pretrained with **masked language modelling** (MLM): 15% of the input tokens are selected, and the model must predict them from the surrounding context. Of the selected tokens, 80% are replaced by a special `[MASK]` token, 10% by a random token and 10% are left unchanged, so that the model cannot rely on `[MASK]` always marking the positions to predict and learns useful representations for *every* position. The loss is computed only at the selected positions (`ignore_index` elsewhere). The bidirectional representations are excellent for classification, tagging and retrieval, which is how Notebook 10 · NLP with Pretrained Transformers (Hugging Face) uses them.

**Encoder–decoder (T5, BART).** The original layout: an encoder reads the source, a decoder with causal self-attention *and* cross-attention to the encoder writes the target. T5 frames every task as text-to-text and pretrains with *span corruption*, a masked-LM variant in which contiguous spans are replaced by sentinel tokens and the decoder reproduces them. This layout is natural for translation and summarisation, where input and output are distinct.

**Decoder-only (GPT, Llama, and most current LLMs).** Causal blocks trained on next-token prediction, as in Section 6. The objective is the simplest, every token in the corpus is a training signal, and the same model handles understanding and generation by *prompting*: the task description and the input become the prefix, the answer is the continuation. This flexibility, together with the discovery that large decoder-only models learn tasks from a few examples in the prompt ("in-context learning", Brown et al., 2020), is why the decoder-only layout has become dominant.

The cell implements the BERT masking procedure on a batch of character windows; Exercise 4 trains a tiny masked LM with it.

In [ ]:
MASK_ID = VOCAB_SIZE                      # one extra token id reserved for [MASK]

def mask_tokens(x, p=0.15, mask_id=MASK_ID, vocab_size=VOCAB_SIZE, generator=None):
    """BERT-style masking: select p of the tokens; replace 80% by [MASK], 10% by a random token, keep 10%.

    Returns (corrupted input, labels) where labels are -100 at unselected positions."""
    labels = x.clone()
    selected = torch.rand(x.shape, generator=generator) < p
    labels[~selected] = -100
    x = x.clone()
    r = torch.rand(x.shape, generator=generator)
    x[selected & (r < 0.8)] = mask_id                                           # 80%: [MASK]
    random_pos = selected & (r >= 0.8) & (r < 0.9)                              # 10%: random token
    x[random_pos] = torch.randint(vocab_size, x.shape, generator=generator)[random_pos]
    return x, labels                                                            # remaining 10%: unchanged


xb, _ = get_batch(train_data, 1, 60)
xb = xb.cpu()
g = torch.Generator().manual_seed(SEED)
x_masked, labels = mask_tokens(xb, generator=g)
render = lambda ids: "".join("_" if int(i) == MASK_ID else itos[int(i)] for i in ids)
print("original :", repr(decode(xb[0].tolist())))
print("corrupted:", repr(render(x_masked[0].tolist())), "  ('_' = [MASK])")
print(f"positions with a prediction target: {(labels[0] != -100).sum().item()} of {xb.shape[1]}")

### 9.2 `nn.TransformerEncoderLayer` and friends

PyTorch ships the block as `nn.TransformerEncoderLayer` (self-attention + MLP), `nn.TransformerDecoderLayer` (adds cross-attention), the stacks `nn.TransformerEncoder` / `nn.TransformerDecoder`, and the full `nn.Transformer`. The arguments map onto Section 5: `d_model`, `nhead`, `dim_feedforward` (the $4d$ MLP width), `dropout`, `activation` (`"relu"` by default; pass `"gelu"` for the GPT/BERT convention), `batch_first=True`, and `norm_first=True` for pre-LN. Masks follow the `nn.MultiheadAttention` convention, `True` = masked out: `src_key_padding_mask` of shape `(batch, T)` for padding, and `mask` of shape `(T, T)` for causal attention, for which `nn.Transformer.generate_square_subsequent_mask(T)` returns the float $0 / -\infty$ version. The cell runs a two-layer pre-LN encoder over a padded batch, both bidirectionally and causally, and checks that padding and causality behave as they should.

In [ ]:
torch.manual_seed(SEED)
layer = nn.TransformerEncoderLayer(d_model=64, nhead=4, dim_feedforward=256, dropout=0.1,
                                   activation="gelu", batch_first=True, norm_first=True)
encoder = nn.TransformerEncoder(layer, num_layers=2, enable_nested_tensor=False).eval()
x = torch.randn(2, 10, 64)
key_padding = torch.zeros(2, 10, dtype=torch.bool)
key_padding[1, 7:] = True                                 # sequence 1 has 3 padded positions (True = masked out)
with torch.no_grad():
    out = encoder(x, src_key_padding_mask=key_padding)
    x_changed = x.clone(); x_changed[1, 8] += 5.0        # perturb a padded position of sequence 1
    out_changed = encoder(x_changed, src_key_padding_mask=key_padding)
    causal_float = nn.Transformer.generate_square_subsequent_mask(10)
    out_causal = encoder(x, mask=causal_float)
    x_future = x.clone(); x_future[0, 9] += 5.0          # perturb the last position of sequence 0
    out_causal_future = encoder(x_future, mask=causal_float)
print("output:", tuple(out.shape))
print("real positions ignore changes at padded positions :", torch.allclose(out[1, :7], out_changed[1, :7], atol=1e-5))
print("causal: positions 0..8 ignore a change at position 9:", torch.allclose(out_causal[0, :9], out_causal_future[0, :9], atol=1e-5))
print("float causal mask, first 3 rows:\n", causal_float[:3, :4])
print("encoder parameters:", sum(p.numel() for p in encoder.parameters()),
      "| formula 2 * (12d^2 + 13d) =", 2 * block_params_formula(64))

The built-in layer has the same parameter count as two of our blocks and honours both masks. It is a perfectly good choice for encoders and for research code; the reason large-model codebases still write their own attention (as we did) is control over the KV-cache, positional schemes like RoPE, and kernel choices.

### 9.3 Where this leads

Everything in Notebook 10 · NLP with Pretrained Transformers (Hugging Face) is an instance of what we built here. DistilBERT is an encoder-only stack trained with masked language modelling, `distilgpt2` and `gpt2` are the `GPT` class with 6 or 12 layers and a byte-pair-encoded vocabulary of 50,257 tokens instead of characters, and fine-tuning, LoRA and instruction tuning are ways of continuing to train these same weights on new objectives. The Vision Transformer of Notebook 07 · Transfer Learning and Modern Vision was the encoder with image patches as tokens, and the diffusion models of Notebook 12 · GANs and Diffusion Models use attention inside their U-Nets. Having implemented attention, masks, positional encodings, the block, the model, the training recipe and decoding from scratch, you can read those models' source code and recognise every line.

## Exercises

**Exercise 1 (warm-up).** Implement a *relative position bias* in the style of T5: a learned table $b \in \mathbb{R}^{(2R+1) \times h}$ indexed by the clipped offset $\operatorname{clip}(j - i, -R, R)$ between key position $j$ and query position $i$, added to the attention scores before the softmax (one scalar per head and offset). Show that the bias matrix is a Toeplitz matrix (constant along diagonals), and demonstrate how to pass it to `F.scaled_dot_product_attention` together with a causal mask as a single float mask.

**Solution to Exercise 1**

A float `attn_mask` is *added* to the scores, so the bias and the $-\infty$ causal entries can be combined into one tensor of shape `(1, h, T, T)` that broadcasts over the batch. Because the table is indexed by $j - i$ only, every diagonal of the resulting matrix holds a single value, and the bias for a window of positions is the same wherever the window sits, which is what makes the scheme translation-invariant and able to handle sequences longer than those seen in training (offsets beyond $R$ share the clipped entry).

In [ ]:
class RelativePositionBias(nn.Module):
    """Learned scalar bias per head and clipped relative offset (T5-style, without bucketing)."""

    def __init__(self, n_heads, max_distance=16):
        super().__init__()
        self.max_distance = max_distance
        self.table = nn.Embedding(2 * max_distance + 1, n_heads)

    def forward(self, T_q, T_k, device=None):
        i = torch.arange(T_q, device=device)[:, None]
        j = torch.arange(T_k, device=device)[None, :]
        offset = (j - i).clamp(-self.max_distance, self.max_distance) + self.max_distance   # (T_q, T_k) in [0, 2R]
        return self.table(offset).permute(2, 0, 1)[None]                                    # (1, h, T_q, T_k)


torch.manual_seed(SEED)
h, T_rel, d_h = 4, 12, 8
rel_bias = RelativePositionBias(h, max_distance=4)
bias = rel_bias(T_rel, T_rel)
print("bias shape:", tuple(bias.shape))
diagonals_constant = all(torch.allclose(torch.diagonal(bias[0, 0], offset=o), torch.diagonal(bias[0, 0], offset=o)[:1].expand(T_rel - abs(o)))
                         for o in range(-T_rel + 1, T_rel))
print("constant along every diagonal (Toeplitz):", diagonals_constant)
print("same bias for a window at positions 0..5 and 6..11:", torch.allclose(bias[0, :, :6, :6], bias[0, :, 6:, 6:]))

q, k, v = (torch.randn(2, h, T_rel, d_h) for _ in range(3))
float_mask = bias.masked_fill(~causal_mask(T_rel), float("-inf"))              # bias + causal mask in one tensor
with torch.no_grad():
    y = F.scaled_dot_product_attention(q, k, v, attn_mask=float_mask)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_h) + float_mask
    y_manual = F.softmax(scores, dim=-1) @ v
print("SDPA with the float mask matches the manual computation:", torch.allclose(y, y_manual, atol=1e-5))

**Exercise 2.** Dropout is a regulariser, so it should help when the model can overfit. Train two copies of the FAST configuration of the GPT on a *small* slice of the corpus (the first 20,000 characters), one with `dropout=0.0` and one with `dropout=0.2`, for the same number of steps, and compare the final training loss (evaluated in `eval` mode on training windows) and validation loss. Interpret the gap.

**Solution to Exercise 2**

On 20k characters the tiny model sees each character dozens of times within the budget and starts to memorise: the training loss keeps falling while the validation loss flattens or rises. Dropout narrows the gap by raising the training loss and, if the run is long enough, lowering the validation loss. With the short `FAST_RUN` budget the effect may be small; the full budget makes it clear. The lesson generalises: dropout (and weight decay) matter when data is scarce relative to model capacity, and modern large models trained for a single epoch on huge corpora often use no dropout at all.

In [ ]:
small_train = train_data[:20_000]
cfg_small = dict(cfg, steps=300 if FAST_RUN else 1500, eval_every=(300 if FAST_RUN else 1500))
results_dropout = {}
for p_drop in [0.0, 0.2]:
    torch.manual_seed(SEED)
    m = GPT(VOCAB_SIZE, cfg["d_model"], cfg["n_heads"], cfg["n_layers"], cfg["block_size"], dropout=p_drop).to(device)
    print(f"dropout {p_drop}:")
    train_gpt(m, small_train, val_data, cfg_small, prompt_idx, sample_steps=())
    tr = estimate_loss(m, small_train, cfg["batch_size"], cfg["block_size"], 10)
    va = estimate_loss(m, val_data, cfg["batch_size"], cfg["block_size"], 10)
    results_dropout[p_drop] = (tr, va)
for p_drop, (tr, va) in results_dropout.items():
    print(f"dropout {p_drop}: train loss (eval mode) {tr:.3f} | val loss {va:.3f} | generalisation gap {va - tr:+.3f}")

**Exercise 3.** Nucleus (top-$p$) sampling is easy to get subtly wrong. Write a slow, obviously correct reference implementation with a Python loop (sort the probabilities, add tokens until the cumulative mass reaches $p$), and test the vectorised `top_p_filter` of Section 7 against it on random distributions. Then check the edge cases explicitly: (a) $p = 1$ keeps every token; (b) when the most likely token alone has probability above $p$, exactly one token is kept; (c) no row is ever fully masked (which would give `nan` after the softmax). Finally show that the tempting one-line variant `remove = cumulative > p` fails case (b).

**Solution to Exercise 3**

The correct rule removes a token when the mass *before* it already exceeds $p$; the crossing token itself stays. The naive rule removes the crossing token, so a distribution whose top token has mass $0.95$ with $p = 0.9$ loses *all* its tokens and the softmax returns `nan`. Ties at exactly $p$ (a uniform distribution and a $p$ that is a multiple of $1/V$) are the only place where the vectorised version can differ from the reference by one token, because floating-point cumulative sums do not land exactly on $p$; both choices are defensible there.

In [ ]:
def nucleus_reference(logits_row, p):
    """Set of kept token ids for one row, computed with an explicit loop."""
    probs = F.softmax(logits_row, dim=-1)
    order = torch.argsort(probs, descending=True)
    kept, mass = [], 0.0
    for idx in order.tolist():
        kept.append(idx)
        mass += probs[idx].item()
        if mass > p:
            break
    return set(kept)

def kept_set(filtered_row):
    return set(torch.nonzero(torch.isfinite(filtered_row)).flatten().tolist())

torch.manual_seed(SEED)
mismatches = 0
for _ in range(200):
    row = torch.randn(30) * 3
    p = random.choice([0.3, 0.5, 0.8, 0.9, 0.95])
    if kept_set(top_p_filter(row[None], p)[0]) != nucleus_reference(row, p):
        mismatches += 1
print(f"(random) vectorised vs reference: {mismatches} mismatches in 200 trials")

row = torch.log(torch.tensor([0.95, 0.03, 0.02]))
print("(a) p = 1.0 keeps every token       :", kept_set(top_p_filter(row[None], 1.0)[0]) == {0, 1, 2})
print("(b) top token 0.95 > p = 0.9 keeps 1:", kept_set(top_p_filter(row[None], 0.9)[0]) == {0})
peaked = torch.full((4, 50), -30.0); peaked[:, 7] = 10.0
finite_rows = torch.isfinite(top_p_filter(peaked, 0.5)).any(dim=-1).all().item()
print("(c) no fully masked rows            :", finite_rows,
      "| softmax has no nan:", not torch.isnan(F.softmax(top_p_filter(peaked, 0.5), dim=-1)).any().item())

def top_p_naive(logits, p):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
    cumulative = F.softmax(sorted_logits, dim=-1).cumsum(dim=-1)
    sorted_logits = sorted_logits.masked_fill(cumulative > p, float("-inf"))     # removes the crossing token
    return torch.full_like(logits, float("-inf")).scatter(-1, sorted_idx, sorted_logits)

naive = top_p_naive(row[None], 0.9)
print("naive variant on case (b): kept", kept_set(naive[0]), "-> softmax nan:", torch.isnan(F.softmax(naive, dim=-1)).any().item())

**Exercise 4.** Train a tiny BERT-style masked language model on the same character data. Use the `Block` class with `causal=False`, an extra `[MASK]` token, the `mask_tokens` function of Section 9, and a loss that ignores unselected positions. Report the accuracy of the model's predictions at masked positions on validation windows, and show a fill-in-the-blank example.

**Solution to Exercise 4**

The model is the `GPT` class with three changes: bidirectional blocks, a vocabulary enlarged by one for `[MASK]`, and a loss over selected positions only. Because every position can see both sides, the masked-character task is easier than next-character prediction *given enough training*: the accuracy at masked positions rises quickly at first (spaces, common letters and word endings are guessed from their neighbours) and keeps improving with the full budget, where most masked characters are recovered. Under `FAST_RUN` the tiny model typically recovers a third or more of the masked characters after a few hundred steps, well above the roughly 15% obtained by always guessing the most frequent character (the space). Note that such a model cannot generate text left to right; its strength is the representation at every position, which Notebook 10 · NLP with Pretrained Transformers (Hugging Face) exploits for classification.

In [ ]:
class TinyBERT(nn.Module):
    """Encoder-only Transformer for character-level masked language modelling."""

    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_len, dropout=0.0):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size + 1, d_model)            # + [MASK]
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, dropout, causal=False) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size + 1)
        self.apply(GPT._init_weights)

    def forward(self, idx, labels=None):
        pos = torch.arange(idx.shape[1], device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)
        return logits, loss


@torch.no_grad()
def masked_accuracy(model, data, batch_size, block_size, n_batches):
    model.eval()
    g = torch.Generator().manual_seed(SEED)
    correct = total = 0
    for _ in range(n_batches):
        x, _ = get_batch(data, batch_size, block_size, generator=g)
        x_m, labels = mask_tokens(x.cpu(), generator=g)
        logits, _ = model(x_m.to(device))
        pred = logits.argmax(dim=-1).cpu()
        sel = labels != -100
        correct += (pred[sel] == labels[sel]).sum().item(); total += sel.sum().item()
    model.train()
    return correct / total


torch.manual_seed(SEED)
bert = TinyBERT(VOCAB_SIZE, cfg["d_model"], cfg["n_heads"], cfg["n_layers"], cfg["block_size"]).to(device)
opt = configure_optimizer(bert, cfg["lr"], cfg["weight_decay"])
STEPS_MLM = 600 if FAST_RUN else 3000
t0 = time.time()
bert.train()
for step in range(1, STEPS_MLM + 1):
    for group in opt.param_groups:
        group["lr"] = lr_at(step - 1, cfg["lr"], cfg["warmup"], STEPS_MLM, cfg["min_lr"])
    x, _ = get_batch(train_data, cfg["batch_size"], cfg["block_size"])
    x_m, labels = mask_tokens(x.cpu())
    _, loss = bert(x_m.to(device), labels.to(device))
    opt.zero_grad(set_to_none=True); loss.backward()
    nn.utils.clip_grad_norm_(bert.parameters(), 1.0); opt.step()
    if step % max(1, STEPS_MLM // 5) == 0 or step == STEPS_MLM:
        print(f"step {step:5d} | MLM loss {loss.item():.3f} | masked-token accuracy (val) "
              f"{masked_accuracy(bert, val_data, cfg['batch_size'], cfg['block_size'], 5):.3f} | {time.time() - t0:5.1f}s")

x, _ = get_batch(val_data, 1, 60)
g = torch.Generator().manual_seed(SEED + 1)
x_m, labels = mask_tokens(x.cpu(), generator=g)
with torch.no_grad():
    bert.eval()
    pred = bert(x_m.to(device))[0].argmax(dim=-1).cpu()
filled = x_m[0].clone(); sel = labels[0] != -100; filled[sel] = pred[0][sel]
print("\noriginal :", repr(decode(x[0].tolist())))
print("masked   :", repr(render(x_m[0].tolist())))
print("filled in:", repr(render(filled.tolist())))       # render, not decode: a prediction could be the [MASK] id
print(f"correct at masked positions: {(pred[0][sel] == labels[0][sel]).sum().item()} of {sel.sum().item()}")

**Exercise 5 (challenge).** In multi-head attention every head has its own keys and values, so the KV-cache holds $2 \times L \times h \times d_h$ values per token. *Multi-query attention* (MQA, Shazeer 2019) shares a single key and value head across all query heads; *grouped-query attention* (GQA, Ainslie et al., 2023) shares each KV head among a group of query heads. Implement a `GroupedQueryAttention` module with `n_kv_heads` key/value heads (so `n_kv_heads = n_heads` is standard MHA and `n_kv_heads = 1` is MQA), verify that its output shape is correct and that it reduces to plain multi-head attention when `n_kv_heads = n_heads`, and compare parameter counts and KV-cache sizes for the notebook's model and for a Llama-2-70B-like configuration ($L = 80$, $d = 8192$, $h = 64$) at a context of 4,096 tokens in float16.

**Solution to Exercise 5**

The key/value projection shrinks from $d \times d$ to $d \times (n_{kv} \cdot d_h)$, and each KV head is repeated (`repeat_interleave`) to serve its group of query heads at attention time; recent PyTorch versions can do the repetition inside `F.scaled_dot_product_attention` with `enable_gqa=True`, but the explicit version below works everywhere. The saving in parameters is minor; the saving in cache is a factor $h / n_{kv}$, which for Llama-2-70B (8 KV heads for 64 query heads) turns a 10 GiB cache per 4k-token sequence into 1.25 GiB, and that is what allows serving many sequences per GPU.

In [ ]:
class GroupedQueryAttention(nn.Module):
    """Causal self-attention with n_kv_heads key/value heads shared among n_heads query heads."""

    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads, self.d_head = n_heads, n_kv_heads, d_model // n_heads
        self.q = nn.Linear(d_model, d_model)
        self.kv = nn.Linear(d_model, 2 * n_kv_heads * self.d_head)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)                  # (B, h, T, d_h)
        k, v = self.kv(x).split(self.n_kv_heads * self.d_head, dim=-1)
        k = k.view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)                       # (B, h_kv, T, d_h)
        v = v.view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        rep = self.n_heads // self.n_kv_heads
        k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)             # serve each group
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))


torch.manual_seed(SEED)
d_model, n_heads = 64, 8
x = torch.randn(2, 16, d_model)
mha_ref = SelfAttention(d_model, n_heads, causal=True).eval()
gqa_full = GroupedQueryAttention(d_model, n_heads, n_kv_heads=n_heads).eval()
with torch.no_grad():                                        # copy the fused QKV weights: GQA with h_kv = h is MHA
    Wq, Wk, Wv = mha_ref.qkv.weight.chunk(3, dim=0)
    bq, bk, bv = mha_ref.qkv.bias.chunk(3, dim=0)
    gqa_full.q.weight.copy_(Wq); gqa_full.q.bias.copy_(bq)
    gqa_full.kv.weight.copy_(torch.cat([Wk, Wv])); gqa_full.kv.bias.copy_(torch.cat([bk, bv]))
    gqa_full.proj.weight.copy_(mha_ref.proj.weight); gqa_full.proj.bias.copy_(mha_ref.proj.bias)
    print("GQA with n_kv_heads = n_heads equals MHA:", torch.allclose(gqa_full(x), mha_ref(x), atol=1e-6))
    for n_kv in [8, 2, 1]:
        m = GroupedQueryAttention(d_model, n_heads, n_kv)
        print(f"  n_kv_heads={n_kv}: output {tuple(m(x).shape)}, parameters {sum(p.numel() for p in m.parameters()):,}")

print()
print(f"{'configuration':>32} | {'attention params/layer':>22} | {'KV-cache per token':>18} | {'cache @ 4096 tokens':>19}")
for name, L, d, h, T_ctx in [("notebook GPT (full run)", 6, 256, 8, 4096), ("Llama-2-70B-like", 80, 8192, 64, 4096)]:
    d_h = d // h
    for label, n_kv in [("MHA", h), ("GQA-8", 8), ("MQA", 1)]:
        params = d * d + d * 2 * n_kv * d_h + d * d
        cache_per_token = 2 * L * n_kv * d_h * 2                          # float16 bytes
        print(f"{name + ' ' + label:>32} | {params / 1e6:19.2f} M | {cache_per_token / 1024:15.1f} KiB | {cache_per_token * T_ctx / 2**30:16.2f} GiB")

## Summary

- Attention is a differentiable dictionary lookup: scores $QK^{\top}/\sqrt{d_k}$, a row-wise softmax, and a weighted average of values. The $\sqrt{d_k}$ scaling keeps the score variance at 1 so that the softmax does not saturate at initialisation.
- Masks set scores to $-\infty$ before the softmax: padding masks hide pad keys, causal masks hide the future and turn a Transformer into an autoregressive model. Watch the inverted `True = masked out` convention of `nn.MultiheadAttention`.
- Multi-head attention runs $h$ lookups in $d/h$ dimensions in parallel via reshapes; our implementation matches `nn.MultiheadAttention` exactly. `F.scaled_dot_product_attention` and FlashAttention compute the same result without materialising the $T \times T$ matrix.
- Attention is permutation-equivariant; sinusoidal encodings (with the linear relative-shift property), learned position tables, or rotary embeddings (relative offsets via rotations of $Q$ and $K$) restore order.
- A pre-LN Transformer block is $x + \text{Attn}(\text{LN}(x))$ followed by $x + \text{MLP}(\text{LN}(x))$; it has $\approx 12d^2$ parameters and costs $\approx 24d^2 + 4Td$ FLOPs per token in the forward pass, which gives the $6N$ training rule when $T \ll 6d$.
- A GPT is token + position embeddings, $L$ causal blocks, a final LayerNorm and a tied LM head, trained with AdamW ($\beta_2 = 0.95$, decoupled weight decay on matrices), warmup + cosine, and gradient clipping; ours learns Shakespeare-like text from characters in minutes.
- Decoding strategies (greedy, temperature, top-$k$, top-$p$, repetition penalty) shape the output as much as the model does; the KV-cache makes generation linear in the context by reusing keys and values that causality guarantees are fixed.
- Loss follows power laws in parameters, data and compute; compute-optimal training uses roughly 20 tokens per parameter (Chinchilla); attention's quadratic cost dominates only at long contexts, where FlashAttention, GQA/MQA and approximations matter.
- Encoder-only (BERT, masked LM), encoder–decoder (T5) and decoder-only (GPT, next-token) models share the same block and differ in masking and objective; `nn.TransformerEncoderLayer` packages the block for you.

## Further reading

- Vaswani, A. et al., 2017. *Attention Is All You Need*. NeurIPS.
- Bahdanau, D., Cho, K. and Bengio, Y., 2015. *Neural Machine Translation by Jointly Learning to Align and Translate*. ICLR.
- Radford, A. et al., 2019. *Language Models are Unsupervised Multitask Learners* (GPT-2). OpenAI technical report.
- Devlin, J. et al., 2019. *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.
- Xiong, R. et al., 2020. *On Layer Normalization in the Transformer Architecture*. ICML.
- Su, J. et al., 2021. *RoFormer: Enhanced Transformer with Rotary Position Embedding*.
- Dao, T. et al., 2022. *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness*. NeurIPS.
- Holtzman, A. et al., 2020. *The Curious Case of Neural Text Degeneration*. ICLR.
- Kaplan, J. et al., 2020. *Scaling Laws for Neural Language Models*; Hoffmann, J. et al., 2022. *Training Compute-Optimal Large Language Models* (Chinchilla).
- Ainslie, J. et al., 2023. *GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints*. EMNLP.
- Karpathy, A., *nanoGPT* and the *Let's build GPT* lecture (2023), a minimal GPT-2 reimplementation on which the structure of Section 6 is modelled.
- Zhang, A., Lipton, Z. C., Li, M. and Smola, A. J., *Dive into Deep Learning*, Chapter 11 (Attention Mechanisms and Transformers).
- PyTorch documentation: `torch.nn.functional.scaled_dot_product_attention`, `torch.nn.MultiheadAttention`, `torch.nn.TransformerEncoderLayer` — https://docs.pytorch.org/docs/stable/